In [1]:
import pandas as pd

In [2]:
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_datetime"])

In [3]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        200000 non-null  int64         
 1   customer_id     200000 non-null  int64         
 2   order_datetime  198067 non-null  datetime64[us]
 3   channel         200000 non-null  str           
 4   status          200000 non-null  str           
dtypes: datetime64[us](1), int64(2), str(2)
memory usage: 9.7 MB


In [4]:
# 1. NaN drop: order_datetime column
o = orders.dropna(subset=["order_datetime"])
o.info()

<class 'pandas.DataFrame'>
Index: 198067 entries, 0 to 199999
Data columns (total 5 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_id        198067 non-null  int64         
 1   customer_id     198067 non-null  int64         
 2   order_datetime  198067 non-null  datetime64[us]
 3   channel         198067 non-null  str           
 4   status          198067 non-null  str           
dtypes: datetime64[us](1), int64(2), str(2)
memory usage: 11.2 MB


In [5]:
# order_datetime 컬럼에서 월 추출
o["month"] = o["order_datetime"].dt.month
o.head()

,order_id,customer_id,order_datetime,channel,status,month
0,25463,1077,2024-06-25 00:17:41,store,delivered,6
1,171088,2998,2024-05-28 19:35:20,web,delivered,5
2,27437,4066,2024-02-14 17:49:14,app,delivered,2
3,98425,3243,2024-05-08 16:37:36,store,delivered,5
4,22325,5331,2024-04-17 20:08:22,app,delivered,4


In [6]:
jan = o[o["month"]==1]  # 1월 추출
feb = o[o["month"]==2]  # 2월 추출
print(f"1월 주문 개수: {jan.shape[0]}, 2월의 주문 개수: {feb.shape[0]}")

1월 주문 개수: 23583, 2월의 주문 개수: 25761


In [7]:
j_f = pd.concat([jan, feb], axis=0) # 위 아래 붙이기
# j_f = pd.concat([jan, feb], axis=1) # 좌우 붙이기
print(f"1월-2월 주문 개수 : {j_f.shape[0]}")

1월-2월 주문 개수 : 49344


In [8]:
jan.head()

,order_id,customer_id,order_datetime,channel,status,month
6,72872,1802,2024-01-04 21:10:14,store,paid,1
10,130357,4823,2024-01-15 12:55:53,store,delivered,1
15,100725,5933,2024-01-09 14:51:51,web,delivered,1
23,116280,1551,2024-01-01 05:20:25,app,paid,1
30,92663,2070,2024-01-09 09:39:16,store,delivered,1


In [9]:
feb.head()

,order_id,customer_id,order_datetime,channel,status,month
2,27437,4066,2024-02-14 17:49:14,app,delivered,2
11,75790,5802,2024-02-10 06:18:54,web,shipped,2
24,137576,2176,2024-02-15 11:45:18,web,delivered,2
57,63194,1496,2024-02-07 23:08:12,web,shipped,2
59,70869,1579,2024-02-07 05:06:50,web,shipped,2


In [10]:
j_f.head()

,order_id,customer_id,order_datetime,channel,status,month
6,72872,1802,2024-01-04 21:10:14,store,paid,1
10,130357,4823,2024-01-15 12:55:53,store,delivered,1
15,100725,5933,2024-01-09 14:51:51,web,delivered,1
23,116280,1551,2024-01-01 05:20:25,app,paid,1
30,92663,2070,2024-01-09 09:39:16,store,delivered,1


In [11]:
j_f.tail()

,order_id,customer_id,order_datetime,channel,status,month
199946,157415,5852,2024-02-13 00:58:49,web,delivered,2
199965,47132,4942,2024-02-10 07:21:27,store,delivered,2
199969,34202,4823,2024-02-28 00:42:22,store,returned,2
199974,1735,2289,2024-02-19 16:41:24,web,delivered,2
199990,162029,2436,2024-02-26 15:05:43,store,shipped,2


In [12]:
# concat
df1 = pd.DataFrame({"A":[1,2], "B":[3,4]})
df2 = pd.DataFrame({"A":[5,6], "B":[7,8]})

In [13]:
df1

,A,B
0,1,3
1,2,4


In [14]:
pd.concat([df1, df2])

,A,B
0,1,3
1,2,4
0,5,7
1,6,8


In [15]:
pd.concat([df1, df2], ignore_index=True)

,A,B
0,1,3
1,2,4
2,5,7
3,6,8


In [16]:
pd.concat([df1,df2], axis=1)

,A,B,A,B
0,1,3,5,7
1,2,4,6,8


In [17]:
df3 = pd.DataFrame({"A":[9],"C":[10]})
df3

,A,C
0,9,10


In [18]:
df1

,A,B
0,1,3
1,2,4


In [19]:
pd.concat([df1,df3], axis=0)

,A,B,C
0,1,3.0,NaN
1,2,4.0,NaN
0,9,NaN,10.0


In [20]:
pd.concat([df1, df3], axis=1)

,A,B,A,C
0,1,3,9.0,10.0
1,2,4,NaN,NaN


In [21]:
# 재구조화: pivot(long -> wide)
# 카테고리, 월별 매출(amount)
customers = pd.read_csv("../data/customers.csv")
products = pd.read_csv("../data/products.csv")
orders = pd.read_csv("../data/orders.csv", parse_dates=["order_datetime"])
items = pd.read_csv("../data/order_items.csv")

In [22]:
# 1. 정제
products["price"] = pd.to_numeric(products["price"], errors="coerce")
products["category"] = products["category"].str.strip()
prod = products.drop_duplicates(subset="product_id")
cust = customers.drop_duplicates(subset="customer_id")
items["unit_price"] = pd.to_numeric(items["unit_price"], errors="coerce")

In [23]:
# 2. 결측 처리
items = items.merge(
    prod[["product_id","price"]], on="product_id", how="left"   # 50만 건.
)
items.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_item_id  500000 non-null  int64  
 1   order_id       500000 non-null  int64  
 2   product_id     500000 non-null  int64  
 3   quantity       500000 non-null  int64  
 4   unit_price     484887 non-null  float64
 5   discount       500000 non-null  float64
 6   price          424080 non-null  float64
dtypes: float64(3), int64(4)
memory usage: 26.7 MB


In [24]:
# NaN 처리 : items["unit_price"] = price
items["unit_price"] = items["unit_price"].fillna(items["price"])    # unit_price 채우려고 price 가져옴.
items.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_item_id  500000 non-null  int64  
 1   order_id       500000 non-null  int64  
 2   product_id     500000 non-null  int64  
 3   quantity       500000 non-null  int64  
 4   unit_price     497676 non-null  float64
 5   discount       500000 non-null  float64
 6   price          424080 non-null  float64
dtypes: float64(3), int64(4)
memory usage: 26.7 MB


In [25]:
# price 컬럼 삭제
# items = items.drop(columns="price")
items.info()

<class 'pandas.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   order_item_id  500000 non-null  int64  
 1   order_id       500000 non-null  int64  
 2   product_id     500000 non-null  int64  
 3   quantity       500000 non-null  int64  
 4   unit_price     497676 non-null  float64
 5   discount       500000 non-null  float64
 6   price          424080 non-null  float64
dtypes: float64(3), int64(4)
memory usage: 26.7 MB


In [26]:
# 순차 병합 full
full = (
    items
    .merge(orders, on="order_id", how="left", validate="m:1")
    .merge(cust, on="customer_id", how="left", validate="m:1")
    .merge(prod, on="product_id", how="left", validate="m:1")
)
full.head()

,order_item_id,order_id,product_id,quantity,unit_price,discount,price_x,customer_id,order_datetime,channel,...,name,gender,birth_date,signup_date,city,email,product_name,category,price_y,cost
0,59256,114625,3,1,20300.0,0.05,20300.0,5614,2024-06-13 16:58:05,app,...,오경하,F,1981-09-22,2022-06-26,인천,user5614@example.com,플러스 잡지,도서,20300.0,10500.0
1,230587,66337,80,2,90000.0,0.45,90000.0,5500,2024-04-02 18:18:21,app,...,신연서,남,1984-07-17,2023-05-21,광주,user5500@example.com,플러스 선반,가구,90000.0,65700.0
2,279813,163343,3,3,20300.0,0.23,20300.0,4823,2024-06-26 06:42:37,store,...,최은경,남,2004-04-09,2023-05-29,부산,user4823@example.com,플러스 잡지,도서,20300.0,10500.0
3,88487,180332,79,1,8600.0,0.38,8600.0,4266,2024-02-02 08:59:27,app,...,권진우,여,2006-10-06,2021-05-17,고양,user4266@example.com,플러스 립스틱,뷰티,8600.0,5900.0
4,39240,174179,232,1,42600.0,0.31,42600.0,5564,2024-06-22 06:41:40,web,...,송수연,여,1986-02-24,2021-05-26,수원,user5564@example.com,프리미엄 립스틱,뷰티,42600.0,31400.0


In [28]:
# 매출
full["amount"] = full["unit_price"] * full["quantity"] * (1-full["discount"])
print(f"통합 테이블 shape: {full.shape}")
print(f"주문 금액 결측 개수 : {full['amount'].isna().sum()}") 

통합 테이블 shape: (500000, 22)
주문 금액 결측 개수 : 2324


In [35]:
# 카테고리, 월별(orders) 매출(order_items) 집계 = full DataFrame
# 키가 되는 주문일자가 NaN 삭제
f = full.dropna(subset=["order_datetime"])
# 월 데이터 추출해서 새로운 컬럼에 대입
f["month"] = f["order_datetime"].dt.month
f.info()

<class 'pandas.DataFrame'>
Index: 495220 entries, 0 to 499999
Data columns (total 23 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   order_item_id   495220 non-null  int64         
 1   order_id        495220 non-null  int64         
 2   product_id      495220 non-null  int64         
 3   quantity        495220 non-null  int64         
 4   unit_price      492912 non-null  float64       
 5   discount        495220 non-null  float64       
 6   price_x         420027 non-null  float64       
 7   customer_id     495220 non-null  int64         
 8   order_datetime  495220 non-null  datetime64[us]
 9   channel         495220 non-null  str           
 10  status          495220 non-null  str           
 11  name            492762 non-null  str           
 12  gender          457883 non-null  str           
 13  birth_date      464203 non-null  str           
 14  signup_date     492762 non-null  str           
 15 

In [45]:
# 재구조화 : long => wide(pivot), wide => long
# f.groupby(["category","month"])     # 카테고리별, 월별 grouping => DataFrameGroupBy
# f.groupby(["category","month"])["amount"]   # DataFrame[[컬럼명, ...]] 컬럼 추출. amount 추출
# reset_index(): category, month index를 컬럼으로 보내라.
long = f.groupby(["category","month"])["amount"].sum()#.reset_index()
long.loc[("가구", 4)]
long

category  month
가구        1        2.092437e+09
          2        2.220471e+09
          3        2.772762e+09
          4        2.984746e+09
          5        3.425288e+09
          6        3.702633e+09
도서        1        5.480889e+08
          2        5.969805e+08
          3        7.559081e+08
          4        7.923411e+08
          5        9.095838e+08
          6        9.909224e+08
뷰티        1        4.360007e+08
          2        4.949566e+08
          3        5.998881e+08
          4        6.489715e+08
          5        7.249700e+08
          6        7.965551e+08
식품        1        2.419964e+08
          2        2.598230e+08
          3        3.241283e+08
          4        3.461965e+08
          5        3.894168e+08
          6        4.297518e+08
의류        1        5.619913e+08
          2        6.423911e+08
          3        8.037083e+08
          4        8.454202e+08
          5        9.538516e+08
          6        1.050135e+09
전자        1        2.857

In [47]:
pd.options.display.float_format="{:,.0f}".format
long = f.groupby(["category","month"])["amount"].sum().reset_index()
long

,category,month,amount
0,가구,1,"2,092,437,367"
1,가구,2,"2,220,471,099"
2,가구,3,"2,772,762,273"
3,가구,4,"2,984,746,284"
4,가구,5,"3,425,288,120"
5,가구,6,"3,702,632,829"
6,도서,1,"548,088,948"
7,도서,2,"596,980,465"
8,도서,3,"755,908,058"
9,도서,4,"792,341,059"


In [ ]:
# pivot : wide
# wide = long.pivot(
#     # argument에 * 있으면 keword argument로 해야 함. so, Error
#     'month', 
#     'category', 
#     'amount'    
# )

wide = long.pivot(
    # keword argument.
    columns = 'month',  # 컬럼에 들어갈 컬럼 지정
    index = 'category', # 인덱스에 들어갈 컬럼 지정
    values = "amount"   # 값 영역에 들어갈 컬럼 지정
)
wide

month,1,2,3,4,5,6
category,,,,,,
가구,"2,092,437,367","2,220,471,099","2,772,762,273","2,984,746,284","3,425,288,120","3,702,632,829"
도서,"548,088,948","596,980,465","755,908,058","792,341,059","909,583,849","990,922,354"
뷰티,"436,000,735","494,956,566","599,888,135","648,971,531","724,970,039","796,555,060"
식품,"241,996,428","259,822,954","324,128,329","346,196,519","389,416,815","429,751,784"
의류,"561,991,327","642,391,079","803,708,285","845,420,235","953,851,647","1,050,134,797"
전자,"2,857,536,408","3,145,439,576","3,913,578,929","4,240,967,749","4,757,509,122","5,238,382,550"


In [ ]:
# melt: wide => long
wide.reset_index()

month,category,1,2,3,4,5,6
0,가구,"2,092,437,367","2,220,471,099","2,772,762,273","2,984,746,284","3,425,288,120","3,702,632,829"
1,도서,"548,088,948","596,980,465","755,908,058","792,341,059","909,583,849","990,922,354"
2,뷰티,"436,000,735","494,956,566","599,888,135","648,971,531","724,970,039","796,555,060"
3,식품,"241,996,428","259,822,954","324,128,329","346,196,519","389,416,815","429,751,784"
4,의류,"561,991,327","642,391,079","803,708,285","845,420,235","953,851,647","1,050,134,797"
5,전자,"2,857,536,408","3,145,439,576","3,913,578,929","4,240,967,749","4,757,509,122","5,238,382,550"


In [59]:
wide.reset_index().melt(
    id_vars="category", # 고정할 식별자 컬럼
    var_name='month', # 이름 변경할 때 사용
    value_name="amount" # 값 영역에 들어갈 컬럼 지정
)

,category,month,amount
0,가구,1,"2,092,437,367"
1,도서,1,"548,088,948"
2,뷰티,1,"436,000,735"
3,식품,1,"241,996,428"
4,의류,1,"561,991,327"
5,전자,1,"2,857,536,408"
6,가구,2,"2,220,471,099"
7,도서,2,"596,980,465"
8,뷰티,2,"494,956,566"
9,식품,2,"259,822,954"
